# Notebook 01 — Descarga y limpieza de la ENOE

Cargo los microdatos del 4T 2025 de la ENOE, pego las tres tablas que necesito (SDEMT, COE1, COE2), construyo las variables del modelo Mincer y dejo guardado un dataset limpio en `data/processed/`. Todo ponderado por factor de expansión desde el primer paso, porque olvidarlo aunque sea una vez rompe los números.

El notebook está pensado para correrse de arriba abajo sin tocar nada, asumiendo que ya descargaste el ZIP del INEGI y lo descomprimiste en `data/raw/enoen_2025_4t/`. Las instrucciones para hacerlo están en la siguiente celda.


## Descarga del INEGI (manual, 5 minutos)

El INEGI no permite descarga programática desde notebooks de terceros — hay que pasar por la página. No hay vuelta. Pasos:

1. Entrar a la página de microdatos de la ENOE:
   <https://www.inegi.org.mx/programas/enoe/15ymas/#microdatos>
2. Buscar el bloque del **4T 2025** y bajar el archivo **"Conjunto de datos"** en formato CSV.
3. Descomprimir el ZIP. Te van a quedar varios CSVs con nombres como `SDEMT425.csv`, `COE1T425.csv`, `COE2T425.csv` (las nomenclaturas cambian de un trimestre a otro).
4. Mover todo el contenido descomprimido a `data/raw/enoen_2025_4t/`.

Una vez ahí, el módulo `enoe_loader.detectar_archivos` encuentra las tablas por patrón sin importar cómo se llamen exactamente.


In [ ]:
# Imports y configuración
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importar el módulo local
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import enoe_loader as enoe

# Configuración pandas para que no recorte columnas en pantalla
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

DATA_RAW = ROOT / "data" / "raw" / "enoen_2025_4t"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Carpeta de datos crudos: {DATA_RAW}")
print(f"Existe: {DATA_RAW.exists()}")


## Detectar las tablas

`detectar_archivos` busca SDEMT, COE1 y COE2 por patrón. Si alguna falta, lanza error claro con los archivos que sí están en la carpeta — útil para diagnosticar problemas de descomprimido.


In [ ]:
archivos = enoe.detectar_archivos(DATA_RAW)
for nombre, ruta in archivos.items():
    print(f"{nombre:6s} → {ruta.name}")


## Cargar las tres tablas

El INEGI guarda los CSVs con encoding `latin-1`, no UTF-8. Si los lees con default vas a ver caracteres rotos en nombres con acentos. El módulo ya lo maneja.


In [ ]:
sdemt = enoe.cargar_tabla(archivos["sdemt"])
coe1  = enoe.cargar_tabla(archivos["coe1"])
coe2  = enoe.cargar_tabla(archivos["coe2"])

print("SDEMT:", sdemt.shape)
print("COE1: ", coe1.shape)
print("COE2: ", coe2.shape)


In [ ]:
# Vistazo rápido a SDEMT — la tabla sociodemográfica
sdemt.head(3)


## Pegar las tablas

Las tres tablas comparten 7 llaves (`cd_a`, `ent`, `con`, `v_sel`, `n_hog`, `h_mud`, `n_ren`). Hago inner join — me quedo solo con personas que aparecen en las tres. Las pérdidas son <1%.


In [ ]:
df = enoe.pegar_enoe(sdemt, coe1, coe2)
print(f"Filas tras pegar: {len(df):,}")
print(f"Columnas: {df.shape[1]}")


## Construir variables del modelo Mincer

`construir_variables_mincer` deja listas las variables que vamos a usar en regresiones (notebook 03 en adelante):

- `mujer` (1/0)
- `anios_escolaridad`, `experiencia`, `experiencia2`
- `ingreso_mensual`, `horas_semanales`, `ingreso_hora`, `log_ingreso_hora`
- `ocupada`, `formal`, `sector`
- `fac_tri` (factor de expansión, renombrado uniformemente)

La función no filtra registros; mete NaN donde corresponde. El filtrado a muestra de análisis lo hago abajo, de manera explícita, para que se vea cuántos registros pierdo en cada paso.


In [ ]:
df = enoe.construir_variables_mincer(df)
df[["mujer","edad","anios_escolaridad","experiencia","ingreso_hora","log_ingreso_hora",
    "ocupada","horas_semanales","fac_tri"]].head(10)


## Filtrar a la muestra de análisis

Restricciones de la muestra para Mincer (las clásicas en economía laboral):

- Población en edad de trabajar: 18–65 años
- Ocupados (`clase2 == 1` en ENOE)
- Con ingreso reportado > 0
- Con horas semanales reportadas > 0
- Sin missing en escolaridad

Voy contando cuántos pierdo en cada filtro para que quede documentado.


In [ ]:
pasos = []

n0 = len(df)
pasos.append(("Inicial", n0, 0))

df1 = df[(df["edad"] >= 18) & (df["edad"] <= 65)]
pasos.append(("Edad 18-65", len(df1), n0 - len(df1)))

df2 = df1[df1["ocupada"] == 1]
pasos.append(("Ocupada == 1", len(df2), len(df1) - len(df2)))

df3 = df2[df2["ingreso_hora"].notna() & (df2["ingreso_hora"] > 0)]
pasos.append(("Ingreso reportado", len(df3), len(df2) - len(df3)))

df4 = df3[df3["horas_semanales"].notna() & (df3["horas_semanales"] > 0)]
pasos.append(("Horas reportadas", len(df4), len(df3) - len(df4)))

df5 = df4[df4["anios_escolaridad"].notna()]
pasos.append(("Escolaridad reportada", len(df5), len(df4) - len(df5)))

resumen = pd.DataFrame(pasos, columns=["Paso", "Registros", "Pérdida"])
resumen["% Pérdida"] = (resumen["Pérdida"] / n0 * 100).round(2)
resumen


In [ ]:
# Renombro el dataset filtrado para claridad
df_m = df5.copy()
print(f"Muestra final para Mincer: {len(df_m):,} registros")
print(f"Población representada (suma de fac_tri): {df_m['fac_tri'].sum():,.0f}")


## Winsorización del ingreso

El ingreso por hora tiene cola larga a la derecha — un puñado de salarios extremos puede mover la regresión. Recorto al percentil 1 y 99 ponderado por factor de expansión. No descarto filas, solo recorto los valores.


In [ ]:
# Antes
antes_min = df_m["ingreso_hora"].min()
antes_max = df_m["ingreso_hora"].max()
antes_p99 = enoe.media_ponderada(
    (df_m["ingreso_hora"] >= df_m["ingreso_hora"].quantile(0.99)).astype(float),
    df_m["fac_tri"],
)

df_m["ingreso_hora_w"] = enoe.winsorizar(df_m["ingreso_hora"], df_m["fac_tri"])
df_m["log_ingreso_hora_w"] = np.log(df_m["ingreso_hora_w"])

print(f"Ingreso por hora antes: min={antes_min:.2f}, max={antes_max:.2f}")
print(f"Ingreso por hora después: min={df_m['ingreso_hora_w'].min():.2f}, "
      f"max={df_m['ingreso_hora_w'].max():.2f}")


## Verificación rápida — la brecha bruta

Antes de cerrar el notebook, calculo la brecha salarial bruta como sanity check. Si me da un número absurdo (negativo, o mayor a 30%), hay un error en los pasos anteriores. La cifra reportada por el INEGI para 2024 fue de ~14% por hora; espero algo en ese rango para el 4T 2025.

**Cuidado clave aquí:** ya estoy ponderando con `fac_tri`. Si no, el número saldría distinto.


In [ ]:
# Medias ponderadas de ingreso por hora, por sexo
hombres = df_m[df_m["mujer"] == 0]
mujeres = df_m[df_m["mujer"] == 1]

media_h = enoe.media_ponderada(hombres["ingreso_hora_w"], hombres["fac_tri"])
media_m = enoe.media_ponderada(mujeres["ingreso_hora_w"], mujeres["fac_tri"])

brecha_pct = (media_h - media_m) / media_h * 100

print(f"Ingreso por hora promedio (hombres): ${media_h:.2f}")
print(f"Ingreso por hora promedio (mujeres): ${media_m:.2f}")
print(f"Brecha bruta: {brecha_pct:.1f}%")


In [ ]:
# La media del LOG (lo que vamos a modelar) es lo que importa para Mincer
log_h = enoe.media_ponderada(hombres["log_ingreso_hora_w"], hombres["fac_tri"])
log_m = enoe.media_ponderada(mujeres["log_ingreso_hora_w"], mujeres["fac_tri"])

print(f"Media log(ingreso_hora) hombres: {log_h:.4f}")
print(f"Media log(ingreso_hora) mujeres: {log_m:.4f}")
print(f"Diferencia (log): {log_h - log_m:.4f}")
print(f"Aproximación a brecha porcentual (log): {(log_h - log_m) * 100:.1f}%")


In [ ]:
# Histograma del log del ingreso por hora, por sexo
fig, ax = plt.subplots(figsize=(9, 5))

for grupo, color, label in [(hombres, "#2c7fb8", "Hombres"),
                             (mujeres, "#d95f0e", "Mujeres")]:
    ax.hist(grupo["log_ingreso_hora_w"],
            bins=60, weights=grupo["fac_tri"],
            alpha=0.55, color=color, label=label, density=True)

ax.axvline(log_h, color="#2c7fb8", linestyle="--", linewidth=1)
ax.axvline(log_m, color="#d95f0e", linestyle="--", linewidth=1)
ax.set_xlabel("log(ingreso por hora)")
ax.set_ylabel("Densidad ponderada")
ax.set_title("Distribución del log del ingreso por hora — ENOE 4T 2025")
ax.legend()
plt.tight_layout()
plt.show()


## Guardar dataset procesado

Lo dejo en parquet — mantiene tipos de datos y es ~5× más chico que CSV. El notebook 02 (EDA) lo lee desde aquí.


In [ ]:
columnas_finales = [
    "fac_tri", "mujer", "edad", "anios_escolaridad", "experiencia", "experiencia2",
    "ingreso_mensual", "horas_semanales",
    "ingreso_hora", "ingreso_hora_w", "log_ingreso_hora", "log_ingreso_hora_w",
    "ocupada", "formal", "sector", "ent",
]
columnas_finales = [c for c in columnas_finales if c in df_m.columns]

df_out = df_m[columnas_finales].copy()
salida = DATA_PROCESSED / "enoen_2025_4t_mincer.parquet"
df_out.to_parquet(salida, index=False)

print(f"Guardado: {salida}")
print(f"Filas: {len(df_out):,} | Columnas: {len(df_out.columns)}")


## Cierre del notebook 01

Lo que dejé listo para el resto del proyecto:

- Dataset `enoen_2025_4t_mincer.parquet` con las variables del modelo Mincer, ya con factor de expansión arrastrado.
- Brecha bruta calculada como verificación contra el dato oficial del INEGI.
- Documentación de cuántos registros se pierden en cada paso del filtrado.

Siguiente paso: **notebook 02 (EDA)** — distribuciones de brecha por escolaridad, sector, edad y entidad federativa, con factor de expansión en cada agregación.


## Hallazgo importante — replanteamiento del proyecto

La brecha bruta arriba sale en ~1.9% por hora. Esperaba algo más cerca de 12–16% (la cifra que reporta el INEGI/OIT). Antes de asumir bug, verifico contra lo que reportan los titulares — que es brecha **mensual**, no por hora.

In [ ]:
# Brecha mensual vs. brecha por hora vs. diferencia en horas
ing_m_h = enoe.media_ponderada(hombres["ingreso_mensual"], hombres["fac_tri"])
ing_m_m = enoe.media_ponderada(mujeres["ingreso_mensual"], mujeres["fac_tri"])
horas_h = enoe.media_ponderada(hombres["horas_semanales"], hombres["fac_tri"])
horas_m = enoe.media_ponderada(mujeres["horas_semanales"], mujeres["fac_tri"])

print(f"Ingreso mensual:    H=${ing_m_h:>8,.0f}   M=${ing_m_m:>8,.0f}   "
      f"brecha={((ing_m_h-ing_m_m)/ing_m_h*100):.1f}%")
print(f"Horas/semana:       H={horas_h:>5.1f}      M={horas_m:>5.1f}      "
      f"diferencia={((horas_h-horas_m)/horas_h*100):.1f}%")
print(f"Ingreso por hora:   H=${media_h:>5.2f}     M=${media_m:>5.2f}     "
      f"brecha={brecha_pct:.1f}%")


La brecha mensual da 23%, consistente con cifras oficiales. La brecha por hora da 1.9%. La diferencia se explica porque las mujeres trabajan 8.4 horas menos a la semana en el mercado remunerado.

Esto cambia la pregunta del proyecto. Modelar log(ingreso_hora) y descubrir que casi no hay brecha es técnicamente correcto pero engañoso en comunicación — la brecha mensual sí existe y sí importa. Mejor modelar log(ingreso_mensual) tratando log(horas) como una de las variables explicativas. Así la descomposición Oaxaca-Blinder dice cuánto de la brecha mensual se explica por horas, cuánto por composición y cuánto queda residual.

Re-genero el dataset procesado agregando `log_ingreso_mensual_w` y `log_horas_w` (winsorizados) para que el notebook 02 los encuentre.

In [ ]:
# Variables nuevas para el encuadre de brecha mensual
df_m["ingreso_mensual_w"] = enoe.winsorizar(df_m["ingreso_mensual"], df_m["fac_tri"])
df_m["horas_w"] = enoe.winsorizar(df_m["horas_semanales"], df_m["fac_tri"])

df_m["log_ingreso_mensual"]   = np.log(df_m["ingreso_mensual"])
df_m["log_ingreso_mensual_w"] = np.log(df_m["ingreso_mensual_w"])
df_m["log_horas"]   = np.log(df_m["horas_semanales"])
df_m["log_horas_w"] = np.log(df_m["horas_w"])

# Re-exportar parquet enriquecido
columnas_finales = [
    "fac_tri", "mujer", "edad", "anios_escolaridad", "experiencia", "experiencia2",
    "ingreso_mensual", "ingreso_mensual_w", "log_ingreso_mensual", "log_ingreso_mensual_w",
    "horas_semanales", "horas_w", "log_horas", "log_horas_w",
    "ingreso_hora", "ingreso_hora_w", "log_ingreso_hora", "log_ingreso_hora_w",
    "ocupada", "formal", "sector", "ent",
]
columnas_finales = [c for c in columnas_finales if c in df_m.columns]
df_out = df_m[columnas_finales].copy()
df_out.to_parquet(DATA_PROCESSED / "enoen_2025_4t_mincer.parquet", index=False)
print(f"Parquet final: {len(df_out):,} filas × {len(df_out.columns)} columnas")


## Debrief del notebook 01

**Lo que aprendí:**

El "aha moment" del notebook fue descubrir que la brecha por hora es chica (1.9%) y la brecha mensual es grande (23%). El paso intermedio — verificar de inmediato el dato oficial antes de asumir bug — me ahorró dos semanas de modelado en la variable equivocada. Lección: cuando un resultado choca con lo esperado, primero verifica si el encuadre es el correcto antes de cuestionar el código.

**Lo que se me resistió:**

El bug del `sex` como string en lugar de numérico. Mi primer pase del módulo asumió que `clase2` y `sex` venían como enteros. Cuando inspeccioné SDEMT vi que vienen como string con espacios en blanco para missing. El bug fue silencioso — no lanzó error, solo asignó todos los registros al grupo de hombres. Quedó documentado en el código con un comentario advirtiendo a futuro yo.

**Para el notebook 02:**

EDA ponderado por factor de expansión, separado por sexo, con tres ejes: distribución de horas (porque ahí está la mitad del problema), distribución de log(ingreso_mensual_w), y heatmap entidad × nivel_esc para localizar dónde la brecha es más alta. Y un primer test de hipótesis para verificar si las diferencias en horas se explican por hijos pequeños (información de COE1 que reservé).